In [4]:
# fix relative imports
import os
cwd = os.path.normpath(os.getcwd())
cwd = cwd.split(os.sep)
find = cwd.index("fidelity-phase-tran")
newdir = f"{os.sep}".join(cwd[:find+1])
os.chdir(newdir)
%load_ext autoreload
%autoreload 2

import types
import gzip
import pickle
import h5py
import numpy as np
from tenpy.tools import hdf5_io

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
strings = ["results/data/ANNNI-1e986d56-dd8c-4201-92bf-30e0e88b9f5e",
        "results/data/ANNNI-2b5cea72-c3c5-4bf9-a1f1-88274f3e4e1e",
        "results/data/ANNNI-3eedcb05-b814-4d26-981d-f024b9309b63",
        "results/data/ANNNI-8e2b8bce-1d27-41d0-a48b-fbce8d4ab8cc",
        "results/data/ANNNI-8f62ad2e-531b-4d81-8eae-b5ba7e5e3a57",
        "results/data/ANNNI-8fad8665-b3af-43a1-be1b-c27fa8b42920",
        "results/data/ANNNI-9d91ce9e-4f61-424a-98bc-eda1abb33d7c",
        "results/data/ANNNI-14c01030-2c38-4304-819f-0b00b5033732",
        "results/data/ANNNI-163e02cf-c141-41c4-9f54-8f5dbeb10b7a",
        "results/data/ANNNI-7902ff06-aac2-4fab-ac2a-2692af269fd4",
        "results/data/ANNNI-010201b6-5ab8-45af-9e50-507c9b609272",
        "results/data/ANNNI-521180e1-d006-4cba-ab76-cfb997f96bb8",
        "results/data/ANNNI-b61ca36d-e8be-427a-83d7-4c630f1f0388",
        "results/data/ANNNI-e13d6adb-870e-4cdf-abdc-8171b4846a50"]

In [13]:
for string in strings:
    with gzip.open(string + ".pkl", 'rb') as f:
        data = pickle.load(f)
    params = data['params']
    l, n = data['l'], data['n']
    gstates = data['gstates']
    stats = data['stats']
    model = data['model_name']

    if isinstance(gstates[0], (types.BuiltinFunctionType, types.BuiltinMethodType)):
        gstates = [gstate() for gstate in gstates]
        print(f"File: {string} was a builtin method or function")

    # Looking at the extent of the parameters explored (all just double checking all is okay!)
    params_extent = np.concatenate([np.min(params, axis=0), np.max(params, axis=0)])
    params_extent = tuple(params_extent[[0, 2, 1, 3]])
    print(params_extent)
    print(np.shape(params))

    # Taking the tensors
    tensor_list =  gstates

    # Convert the list to a 64x64 nested list
    matrix_64x64 = [tensor_list[i*64:(i+1)*64] for i in range(64)]
    # Flip the nested list vertically
    flipped_matrix = matrix_64x64[::-1]
    # Flatten the nested list back into a single list
    corrected_list = [item for sublist in flipped_matrix for item in sublist]

    # Creating data dictionary to be saved 
    data_h5 = {#"gstates": gstates,  # list of MPS - tensors 
            "gstates": corrected_list, # Flipped list
            "params_extent": params_extent, # (2, N) evenly spaced - in future 
            "n": data["n"],
            "l": data["l"], # Number of qubits (L)
            "dmrg_params": data["dmrg_params"], # DMRG params used in calculation - converting? hashtable 
            "info": {"model_type":{model},}, # Additional info 
            "times": np.array(data["stats"]["times"]), # Times for DMRG } # params swept 
    }

    # Saving your file, make sure to change the name to the name you want to store your file in! 
    filename_to_save = f"{string}.h5"
    with h5py.File(filename_to_save, 'w') as f:
        hdf5_io.save_to_hdf5(f, data_h5)
        print(f"File: {string} saved in hdf5")


KeyboardInterrupt: 

## Check metadata of ground states

In [15]:
import os

# Specify the directory containing your files
directory = 'results/data'

# List all files in the directory
files = os.listdir(directory)

# Filter for .pkl and .h5 files
pkl_and_h5_files = [os.path.join(directory, file) for file in files if file.endswith('.pkl') or file.endswith('.h5')]

# Print the list of file names
for file in pkl_and_h5_files:
    print(f"File: {file} has these metadata:")
    if file.endswith('.pkl'):
        with gzip.open(file, 'rb') as f:
            data = pickle.load(f)
            params = data['params']
            l, n = data['l'], data['n']
            gstates = data['gstates']
            stats = data['stats']
            # model = data['model_name']

    if file.endswith('.h5'):
        with h5py.File(file, "r") as f:
            data = hdf5_io.load_from_hdf5(f)
            params_extent = data['params_extent']
            l, n = data['l'], data['n']
            gstates = data['gstates']
            model = data['info']['model_type']
            times = data['times']

    print(f"Model: {model}, Chain length: {l}, points in the grid per side: {n}")
    print(f"Param_extent: {params_extent}")
    print(f"\n#############################\n\n")

File: results/data/ANNNI-b61ca36d-e8be-427a-83d7-4c630f1f0388.pkl has these metadata:
Model: ANNNI, Chain length: 10, points in the grid per side: 64
Param_extent: (0.1, 1.5, 0.1, 1.5)

#############################


File: results/data/model_annni-4a927297-b40e-49b7-8658-d2c0077a289c.pkl has these metadata:
Model: ANNNI, Chain length: 20, points in the grid per side: 16
Param_extent: (0.1, 1.5, 0.1, 1.5)

#############################


File: results/data/Cluster-93e2a6ec-d847-4007-896f-87dfc8afff13.h5 has these metadata:
Model: ANNNI, Chain length: 10, points in the grid per side: 10
Param_extent: (0.1, 1.5, 0.1, 1.5)

#############################


File: results/data/ANNNI-7474d9df-a970-4b6c-8fb7-3578e77c5d97.pkl has these metadata:
Model: ANNNI, Chain length: 20, points in the grid per side: 10
Param_extent: (0.1, 1.5, 0.1, 1.5)

#############################


File: results/data/model_annni-54ae4275-efa9-493a-bdf4-b7c60faa59f5.pkl has these metadata:
Model: ANNNI, Chain length: 1